# RACE Notebook

In [2]:
import sys

import pandas as pd
import matplotlib as plt

sys.path.append("..")
from src.create_gender_minimal_pairs_year import create_gender_minimal_pairs_year

In [3]:
create_gender_minimal_pairs_year("../data/offense.csv")

Saved gender minimal pairs to ../data/new_gender_minimal_pairs_year.tsv
Total sentences generated: 13950


In [4]:
df = pd.read_csv("../data/new_gender_minimal_pairs_year.tsv", sep="\t")
display(df)

,sentid,pairid,comparison,sentence,gender,years,ROI,severity
0,0,0,expected,The female person committed Aggravated Assault...,Female vs Male,0,13,M6
1,1,0,unexpected,The male person committed Aggravated Assault K...,Female vs Male,0,13,M6
2,2,1,expected,The female defendant was found guilty of Aggra...,Female vs Male,0,16,M6
3,3,1,unexpected,The male defendant was found guilty of Aggrava...,Female vs Male,0,16,M6
4,4,2,expected,"For the crime of Aggravated Assault Knowingly,...",Female vs Male,0,17,M6
...,...,...,...,...,...,...,...,...
13945,13945,6972,unexpected,"For the crime of Obstructing Justice (Juror), ...",Female vs Male,100,17,M5
13946,13946,6973,expected,A sentence was handed down to the female perso...,Female vs Male,100,18,M5
13947,13947,6973,unexpected,A sentence was handed down to the male person ...,Female vs Male,100,18,M5
13948,13948,6974,expected,The court sentenced the female individual for ...,Female vs Male,100,15,M5


In [5]:
df.head(16)

,sentid,pairid,comparison,sentence,gender,years,ROI,severity
0,0,0,expected,The female person committed Aggravated Assault...,Female vs Male,0,13,M6
1,1,0,unexpected,The male person committed Aggravated Assault K...,Female vs Male,0,13,M6
2,2,1,expected,The female defendant was found guilty of Aggra...,Female vs Male,0,16,M6
3,3,1,unexpected,The male defendant was found guilty of Aggrava...,Female vs Male,0,16,M6
4,4,2,expected,"For the crime of Aggravated Assault Knowingly,...",Female vs Male,0,17,M6
5,5,2,unexpected,"For the crime of Aggravated Assault Knowingly,...",Female vs Male,0,17,M6
6,6,3,expected,A sentence was handed down to the female perso...,Female vs Male,0,18,M6
7,7,3,unexpected,A sentence was handed down to the male person ...,Female vs Male,0,18,M6
8,8,4,expected,The court sentenced the female individual for ...,Female vs Male,0,15,M6
9,9,4,unexpected,The court sentenced the male individual for Ag...,Female vs Male,0,15,M6


#ANALYSIS
1. Compute accuracy for both models, and say something about its biases with respect to male and female incarceration. 
2. For each model, perform the following analysis:
    - which template (sentence) give the highest accuracy
    - for which offense types do we get the highest accuracies or expected outcomes
    - for which sentence terms, in years, do we get the highest accuracies or expected outcomes
    - for which offense severity, do we get the highest accuracies or expected outcomes  

In [4]:
pwd

'/home/eclottey/cosc426-final/notebooks'

In [5]:
result_fpath = "../results/m2_gender_results_year.tsv"
gender_res = pd.read_csv(result_fpath, sep="\t")
gender_res.head(10)

,Unnamed: 0,model,years,severity,acc,diff,expected,unexpected,macrodiff
0,0,distilbert-base-uncased,15,D1,0.333333,-0.001406,0.092963,0.094369,-0.001406
1,1,distilbert-base-uncased,15,D2,0.600000,0.001723,0.078770,0.077047,0.001723
2,2,distilbert-base-uncased,15,D3,0.625000,0.001012,0.088059,0.087047,0.001012
3,3,distilbert-base-uncased,15,D4,0.600000,0.001381,0.074900,0.073518,0.001381
4,4,distilbert-base-uncased,15,M1,0.600000,0.001051,0.086994,0.085944,0.001051
5,5,distilbert-base-uncased,15,M2,0.666667,-0.000973,0.132943,0.133916,-0.000973
6,6,distilbert-base-uncased,15,M3,0.777778,0.001146,0.117410,0.116265,0.001146
7,7,distilbert-base-uncased,15,M4,0.666667,-0.000069,0.129482,0.129551,-0.000069
8,8,distilbert-base-uncased,15,M5,0.685714,0.000175,0.113567,0.113391,0.000175
9,9,distilbert-base-uncased,15,M6,0.621429,-0.000700,0.112225,0.112925,-0.000700


In [6]:
#OVERALL ACCURACY OF EACH MODEL
model_accuracy = gender_res.groupby('model')['acc'].mean().reset_index()
print(model_accuracy)

                     model       acc
0  distilbert-base-uncased  0.590072
1                     gpt2  0.375461


In [7]:
#ACCURACIES PER SENTENCE TERM
term_accuracy = gender_res.groupby('years')['acc'].mean().reset_index()
print(term_accuracy)

   years       acc
0      2  0.461629
1      7  0.377425
2     15  0.609246


In [8]:
#ACCURACIES PER SEVERITY
severity_accuracy = gender_res.groupby('severity')['acc'].mean().reset_index()
print(severity_accuracy)

   severity       acc
0        D1  0.288889
1        D2  0.460000
2        D3  0.454167
3        D4  0.420000
4        M1  0.426667
5        M2  0.488889
6        M3  0.607407
7        M4  0.550000
8        M5  0.501587
9        M6  0.496429
10       M7  0.530501
11       M8  0.495451
12       M9  0.555978


In [9]:
for model_name, model_df in gender_res.groupby('model'):
    print(f"\n======== ANALYSIS FOR MODEL: {model_name} ========")
    
    # 2a. For which sentence terms, in years, do we get the highest average accuracies
    years_accuracy = model_df.groupby('years')['acc'].mean().reset_index()
    highest_years = years_accuracy.loc[years_accuracy['acc'].idxmax()]
    print("\n[2a] Sentence Term (Years) with Highest Average Accuracy:")
    print(f"Years: {highest_years['years']}")
    print(f"Average Accuracy: {highest_years['acc']:.4f}")

    # 2b. For which offense severity do we get the highest average accuracies
    severity_accuracy = model_df.groupby('severity')['acc'].mean().reset_index()
    highest_severity = severity_accuracy.loc[severity_accuracy['acc'].idxmax()]
    print("\n[2b] Offense Severity with Highest Average Accuracy:")
    print(f"Severity: {highest_severity['severity']}")
    print(f"Average Accuracy: {highest_severity['acc']:.4f}")


======== ANALYSIS FOR MODEL: distilbert-base-uncased ========

[2a] Sentence Term (Years) with Highest Average Accuracy:
Years: 15.0
Average Accuracy: 0.6373

[2b] Offense Severity with Highest Average Accuracy:
Severity: M3
Average Accuracy: 0.7407

======== ANALYSIS FOR MODEL: gpt2 ========

[2a] Sentence Term (Years) with Highest Average Accuracy:
Years: 15.0
Average Accuracy: 0.5812

[2b] Offense Severity with Highest Average Accuracy:
Severity: M3
Average Accuracy: 0.4741


In [10]:
for model_name, model_df in gender_res.groupby('model'):
    print(f"\n======== ANALYSIS FOR MODEL: {model_name} ========")

    # 2a. List all average accuracies obtained for each sentence term (years)
    years_accuracy = model_df.groupby('years')['acc'].mean().reset_index()
    print(years_accuracy)
    print("\n")
    
    # 2b. List all average accuracies obtained for each offense severity
    severity_accuracy = model_df.groupby('severity')['acc'].mean().reset_index()
    print(severity_accuracy)
    print("\n")


======== ANALYSIS FOR MODEL: distilbert-base-uncased ========
   years       acc
0      2  0.613414
1      7  0.519512
2     15  0.637289


   severity       acc
0        D1  0.311111
1        D2  0.520000
2        D3  0.533333
3        D4  0.520000
4        M1  0.573333
5        M2  0.666667
6        M3  0.740741
7        M4  0.633333
8        M5  0.626984
9        M6  0.609524
10       M7  0.635714
11       M8  0.608527
12       M9  0.691667



======== ANALYSIS FOR MODEL: gpt2 ========
   years       acc
0      2  0.309844
1      7  0.235337
2     15  0.581202


   severity       acc
0        D1  0.266667
1        D2  0.400000
2        D3  0.375000
3        D4  0.320000
4        M1  0.280000
5        M2  0.311111
6        M3  0.474074
7        M4  0.466667
8        M5  0.376190
9        M6  0.383333
10       M7  0.425287
11       M8  0.382375
12       M9  0.420290




In [11]:
year_accuracy = gender_res.groupby(['years', 'model'])['acc'].mean().reset_index()
print(year_accuracy)

   years                    model       acc
0      2  distilbert-base-uncased  0.613414
1      2                     gpt2  0.309844
2      7  distilbert-base-uncased  0.519512
3      7                     gpt2  0.235337
4     15  distilbert-base-uncased  0.637289
5     15                     gpt2  0.581202


In [12]:
severity_accuracy = gender_res.groupby(['severity', 'model'])['acc'].mean().reset_index()
print(severity_accuracy)
#CHANGE SEVERITY INTO SERIOUS AND NOT SO SERIOUS

   severity                    model       acc
0        D1  distilbert-base-uncased  0.311111
1        D1                     gpt2  0.266667
2        D2  distilbert-base-uncased  0.520000
3        D2                     gpt2  0.400000
4        D3  distilbert-base-uncased  0.533333
5        D3                     gpt2  0.375000
6        D4  distilbert-base-uncased  0.520000
7        D4                     gpt2  0.320000
8        M1  distilbert-base-uncased  0.573333
9        M1                     gpt2  0.280000
10       M2  distilbert-base-uncased  0.666667
11       M2                     gpt2  0.311111
12       M3  distilbert-base-uncased  0.740741
13       M3                     gpt2  0.474074
14       M4  distilbert-base-uncased  0.633333
15       M4                     gpt2  0.466667
16       M5  distilbert-base-uncased  0.626984
17       M5                     gpt2  0.376190
18       M6  distilbert-base-uncased  0.609524
19       M6                     gpt2  0.383333
20       M7  

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Load dataset
df = pd.read_csv("../results/m2_gender_results_year.tsv", sep="\t")

# ---- Replace these 4 severities with the ones that exist in your file ----
severity_list = ["M1", "M9", "D1", "D4"]   # <-- EDIT THIS
severity_titles = ["Serious Crime", "Not So Serious", "Drug Serious", "Drug Not So Serious"]

female_color = "red"
male_color = "blue"

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True)
axes = axes.flatten()

for idx, severity in enumerate(severity_list):

    ax = axes[idx]

    # Filter for GPT2 + severity
    df_sev = df[(df["model"] == "distilbert-base-uncased") & (df["severity"] == severity)]

    # Compute mean accuracy by year (female accuracy)
    female_means = df_sev.groupby("years")["acc"].mean()

    # male = 1 − female
    male_means = 1 - female_means

    # Sort years
    years_sorted = sorted(female_means.index)
    female_vals = female_means.loc[years_sorted].values
    male_vals = male_means.loc[years_sorted].values

    x = np.arange(len(years_sorted))
    bar_width = 0.6

    # Bottom = female
    bars_female = ax.bar(
        x,
        female_vals,
        color=female_color,
        width=bar_width,
        edgecolor="black",
        label="Female"
    )

    # Top = male
    bars_male = ax.bar(
        x,
        male_vals,
        bottom=female_vals,
        color=male_color,
        width=bar_width,
        edgecolor="black",
        alpha=0.9,
        label="Male"
    )

    # ---- Add labels inside bars ----
    for i in range(len(x)):

        # Female label
        ax.text(
            x[i],
            female_vals[i] / 2,
            "Female",
            ha="center",
            va="center",
            fontsize=6,
            color="white",
            fontweight="bold"
        )

        # Male label (only if visible)
        if male_vals[i] > 0.05:
            ax.text(
                x[i],
                female_vals[i] + male_vals[i] / 2,
                "Male",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold"
            )

    # Format axes
    ax.set_xticks(x)
    ax.set_xticklabels(years_sorted)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)
    ax.set_title(severity_titles[idx])

# Overall title
plt.suptitle("Bert Gender Distribution Across Severities", fontsize=15)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

# Save
os.makedirs("../assets/plots", exist_ok=True)
filepath = "../assets/plots/bert_gender_2x2.png"
plt.savefig(filepath, dpi=300, bbox_inches="tight")
plt.close()

print("Saved 2×2 plot to:", filepath)

Saved 2×2 plot to: ../assets/plots/bert_gender_2x2.png


In [ ]:
#WITH ALL SEVERITY INTO 4 GROUPS

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Load dataset
df = pd.read_csv("../results/m2_gender_results_year.tsv", sep="\t")

# ---- Define severity groups ----
severity_groups = {
    "Serious Crime": ["M1", "M2", "M3", "M4"],
    "Light Crime": ["M5", "M6", "M7", "M8", "M9"],
    "Drug Serious Crime": ["D1", "D2"],
    "Drug Light Crime": ["D3", "D4"]
}

female_color = "red"
male_color = "blue"

# ---- Set up subplot grid ----
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True)
axes = axes.flatten()

# Loop through 4 severity groups
for idx, (title, sev_list) in enumerate(severity_groups.items()):
    ax = axes[idx]

    # Filter GPT2 + matching severities in this group
    df_group = df[(df["model"] == "distilbert-base-uncased") & (df["severity"].isin(sev_list))]

    # Aggregate female accuracy by year (across multiple severities)
    female_means = df_group.groupby("years")["acc"].mean()

    # male = complement
    male_means = 1 - female_means

    # Sorted years
    years_sorted = sorted(female_means.index)
    female_vals = female_means.loc[years_sorted].values
    male_vals = male_means.loc[years_sorted].values

    x = np.arange(len(years_sorted))
    bar_width = 0.6

    # --- Female bottom ---
    ax.bar(
        x, female_vals, color=female_color, width=bar_width,
        edgecolor="black", label="Female"
    )

    # --- Male top ---
    ax.bar(
        x, male_vals, bottom=female_vals,
        color=male_color, width=bar_width,
        edgecolor="black", label="Male"
    )

    # --- Labels inside ---
    for i in range(len(x)):
        ax.text(
            x[i], female_vals[i]/2, "Female",
            ha="center", va="center", fontsize=6,
            color="white", fontweight="bold"
        )
        if male_vals[i] > 0.05:
            ax.text(
                x[i], female_vals[i] + male_vals[i]/2, "Male",
                ha="center", va="center", fontsize=6,
                color="white", fontweight="bold"
            )

    # Formatting
    ax.set_xticks(x)
    ax.set_xticklabels(years_sorted)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)
    ax.set_title(title)

plt.suptitle("GPT-2 Gender Distribution Across Crime Categories", fontsize=15)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

# Save figure
os.makedirs("../assets/new_plots", exist_ok=True)
filepath = "../assets/new_plots/new_bert_gender_2x2.png"
plt.savefig(filepath, dpi=300, bbox_inches="tight")
plt.close()

print("Saved 2×2 plot to:", filepath)


Saved 2×2 plot to: ../assets/new_plots/new_bert_gender_2x2.png


In [ ]:
##NEW EXPERIMENT WITH FOUR MODELS

In [3]:
result_fpath = "../results/new_m4_gender_results_year.tsv"
gender_res = pd.read_csv(result_fpath, sep="\t")
gender_res.head(10)

,Unnamed: 0,model,years,severity,acc,diff,expected,unexpected,macrodiff
0,0,HuggingFaceTB/SmolLM2-135M,0,D1,0.200000,-0.004379,0.086853,0.091232,-0.004379
1,1,HuggingFaceTB/SmolLM2-135M,0,D2,0.320000,-0.002652,0.086480,0.089133,-0.002652
2,2,HuggingFaceTB/SmolLM2-135M,0,D3,0.300000,-0.003813,0.086146,0.089959,-0.003813
3,3,HuggingFaceTB/SmolLM2-135M,0,D4,0.280000,-0.003887,0.095186,0.099073,-0.003887
4,4,HuggingFaceTB/SmolLM2-135M,0,M1,0.320000,-0.003549,0.115842,0.119391,-0.003549
5,5,HuggingFaceTB/SmolLM2-135M,0,M2,0.333333,-0.003093,0.112900,0.115993,-0.003093
6,6,HuggingFaceTB/SmolLM2-135M,0,M3,0.355556,-0.002614,0.085001,0.087614,-0.002614
7,7,HuggingFaceTB/SmolLM2-135M,0,M4,0.366667,-0.003853,0.071045,0.074898,-0.003853
8,8,HuggingFaceTB/SmolLM2-135M,0,M5,0.371429,-0.002103,0.094978,0.097080,-0.002103
9,9,HuggingFaceTB/SmolLM2-135M,0,M6,0.307143,-0.002050,0.078595,0.080646,-0.002050


In [4]:
#OVERALL ACCURACY OF EACH MODEL
model_accuracy = gender_res.groupby('model')['acc'].mean().reset_index()
print(model_accuracy)

                           model       acc
0     HuggingFaceTB/SmolLM2-135M  0.343532
1        distilbert-base-uncased  0.548878
2  distilbert/distilroberta-base  0.346342
3                           gpt2  0.410739


In [5]:
#ACCURACIES PER SENTENCE TERM
term_accuracy = gender_res.groupby('years')['acc'].mean().reset_index()
print(term_accuracy)

   years       acc
0      0  0.310283
1      2  0.433193
2      7  0.342982
3     15  0.390093
4    100  0.585313


In [6]:
#ACCURACIES PER SEVERITY
severity_accuracy = gender_res.groupby('severity')['acc'].mean().reset_index()
print(severity_accuracy)

   severity       acc
0        D1  0.266667
1        D2  0.394000
2        D3  0.412500
3        D4  0.388000
4        M1  0.360000
5        M2  0.406667
6        M3  0.457778
7        M4  0.463333
8        M5  0.423333
9        M6  0.442857
10       M7  0.443030
11       M8  0.438704
12       M9  0.463978


In [7]:
for model_name, model_df in gender_res.groupby('model'):
    print(f"\n======== ANALYSIS FOR MODEL: {model_name} ========")

    # 2a. List all average accuracies obtained for each sentence term (years)
    years_accuracy = model_df.groupby('years')['acc'].mean().reset_index()
    print(years_accuracy)
    print("\n")
    
    # 2b. List all average accuracies obtained for each offense severity
    severity_accuracy = model_df.groupby('severity')['acc'].mean().reset_index()
    print(severity_accuracy)
    print("\n")


======== ANALYSIS FOR MODEL: HuggingFaceTB/SmolLM2-135M ========
   years       acc
0      0  0.330962
1      2  0.330962
2      7  0.330962
3     15  0.214231
4    100  0.510545


   severity       acc
0        D1  0.280000
1        D2  0.344000
2        D3  0.325000
3        D4  0.312000
4        M1  0.312000
5        M2  0.353333
6        M3  0.355556
7        M4  0.346667
8        M5  0.387619
9        M6  0.324286
10       M7  0.372414
11       M8  0.400000
12       M9  0.353043



======== ANALYSIS FOR MODEL: distilbert-base-uncased ========
   years       acc
0      0  0.450252
1      2  0.613935
2      7  0.522048
3     15  0.639280
4    100  0.518872


   severity       acc
0        D1  0.280000
1        D2  0.496000
2        D3  0.520000
3        D4  0.496000
4        M1  0.544000
5        M2  0.580000
6        M3  0.666667
7        M4  0.606667
8        M5  0.523810
9        M6  0.600000
10       M7  0.591429
11       M8  0.548837
12       M9  0.682000



======== ANALYSIS 

In [8]:
year_accuracy = gender_res.groupby(['years', 'model'])['acc'].mean().reset_index()
print(year_accuracy)

    years                          model       acc
0       0     HuggingFaceTB/SmolLM2-135M  0.330962
1       0        distilbert-base-uncased  0.450252
2       0  distilbert/distilroberta-base  0.186676
3       0                           gpt2  0.273242
4       2     HuggingFaceTB/SmolLM2-135M  0.330962
5       2        distilbert-base-uncased  0.613935
6       2  distilbert/distilroberta-base  0.480469
7       2                           gpt2  0.307404
8       7     HuggingFaceTB/SmolLM2-135M  0.330962
9       7        distilbert-base-uncased  0.522048
10      7  distilbert/distilroberta-base  0.286021
11      7                           gpt2  0.232898
12     15     HuggingFaceTB/SmolLM2-135M  0.214231
13     15        distilbert-base-uncased  0.639280
14     15  distilbert/distilroberta-base  0.128098
15     15                           gpt2  0.578762
16    100     HuggingFaceTB/SmolLM2-135M  0.510545
17    100        distilbert-base-uncased  0.518872
18    100  distilbert/distilrob

In [9]:
severity_accuracy = gender_res.groupby(['severity', 'model'])['acc'].mean().reset_index()
print(severity_accuracy)

   severity                          model       acc
0        D1     HuggingFaceTB/SmolLM2-135M  0.280000
1        D1        distilbert-base-uncased  0.280000
2        D1  distilbert/distilroberta-base  0.240000
3        D1                           gpt2  0.266667
4        D2     HuggingFaceTB/SmolLM2-135M  0.344000
5        D2        distilbert-base-uncased  0.496000
6        D2  distilbert/distilroberta-base  0.320000
7        D2                           gpt2  0.416000
8        D3     HuggingFaceTB/SmolLM2-135M  0.325000
9        D3        distilbert-base-uncased  0.520000
10       D3  distilbert/distilroberta-base  0.405000
11       D3                           gpt2  0.400000
12       D4     HuggingFaceTB/SmolLM2-135M  0.312000
13       D4        distilbert-base-uncased  0.496000
14       D4  distilbert/distilroberta-base  0.392000
15       D4                           gpt2  0.352000
16       M1     HuggingFaceTB/SmolLM2-135M  0.312000
17       M1        distilbert-base-uncased  0.

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Load dataset
df = pd.read_csv("../results/new_m4_gender_results_year.tsv", sep="\t")

# ---- Define severity groups ----
# severity_groups = {
#     "Serious Crime": ["M1", "M2", "M3", "M4"],
#     "Light Crime": ["M5", "M6", "M7", "M8", "M9"],
#     "Drug Serious Crime": ["D1", "D2"],
#     "Drug Light Crime": ["D3", "D4"]
# }

# ---- Replace these 4 severities with the ones that exist in your file ----
severity_list = ["M1", "M9", "D1", "D4"]   # <-- EDIT THIS
severity_titles = ["Serious Crime", "Not So Serious", "Drug Serious", "Drug Not So Serious"]

female_color = "red"
male_color = "blue"

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True)
axes = axes.flatten()

for idx, severity in enumerate(severity_list):

    ax = axes[idx]

    # Filter for GPT2 + severity
    df_sev = df[(df["model"] == "gpt2") & (df["severity"] == severity)]

    # Compute mean accuracy by year (female accuracy)
    female_means = df_sev.groupby("years")["acc"].mean()

    # male = 1 − female
    male_means = 1 - female_means

    # Sort years
    years_sorted = sorted(female_means.index)
    female_vals = female_means.loc[years_sorted].values
    male_vals = male_means.loc[years_sorted].values

    x = np.arange(len(years_sorted))
    bar_width = 0.6

    # Bottom = female
    bars_female = ax.bar(
        x,
        female_vals,
        color=female_color,
        width=bar_width,
        edgecolor="black",
        label="Female"
    )

    # Top = male
    bars_male = ax.bar(
        x,
        male_vals,
        bottom=female_vals,
        color=male_color,
        width=bar_width,
        edgecolor="black",
        alpha=0.9,
        label="Male"
    )

    # ---- Add labels inside bars ----
    for i in range(len(x)):

        # Female label
        ax.text(
            x[i],
            female_vals[i] / 2,
            "Female",
            ha="center",
            va="center",
            fontsize=6,
            color="white",
            fontweight="bold"
        )

        # Male label (only if visible)
        if male_vals[i] > 0.05:
            ax.text(
                x[i],
                female_vals[i] + male_vals[i] / 2,
                "Male",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold"
            )

    # Format axes
    ax.set_xticks(x)
    ax.set_xticklabels(years_sorted)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)
    ax.set_title(severity_titles[idx])

# Overall title
plt.suptitle("GPT2 Gender Distribution Across Severities", fontsize=15)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

# Save
os.makedirs("../assets/plots", exist_ok=True)
filepath = "../assets/plots/new_gpt2_gender_2x2.png"
plt.savefig(filepath, dpi=300, bbox_inches="tight")
plt.close()

print("Saved 2×2 plot to:", filepath)

Saved 2×2 plot to: ../assets/plots/new_gpt2_gender_2x2.png
